In [0]:
# Create widgets
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("load_type", "", "Load Type")
dbutils.widgets.text("merge_keys", "", "Merge Keys")

# Get widget values into variables
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
load_type = dbutils.widgets.get("load_type")
merge_keys = dbutils.widgets.get("merge_keys")

In [0]:
from pyspark.sql.functions import current_timestamp

df = spark.read.table(f"mano_adb_outlook.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'").withColumn("DateAdded", current_timestamp()).withColumn("DateUpdated", current_timestamp())

In [0]:
from delta.tables import DeltaTable

target_table = f"mano_adb_outlook.silver.{table_name}"

if load_type == "FULL":
    df.write.mode("overwrite").saveAsTable(target_table)

elif load_type == "APPEND":
    df.write.mode("append").saveAsTable(target_table)

elif load_type == "MERGE":
    # Build merge condition from comma-separated merge_keys
    keys = [k.strip() for k in merge_keys.split(",")]

    # Validate that all merge keys exist in the source DataFrame
    missing_keys = [k for k in keys if k not in df.columns]
    if missing_keys:
        raise ValueError(f"Merge keys not found in bronze table '{table_name}': {missing_keys}. Available columns: {df.columns}")

    merge_condition = " AND ".join([f"target.`{k}` = source.`{k}`" for k in keys])

    if spark.catalog.tableExists(target_table):
        delta_table = DeltaTable.forName(spark, target_table)

        # Update all columns except DateAdded (preserve original insert timestamp)
        update_set = {c: f"source.`{c}`" for c in df.columns if c != "DateAdded"}
        insert_values = {c: f"source.`{c}`" for c in df.columns}

        delta_table.alias("target").merge(
            df.alias("source"),
            merge_condition
        ).whenMatchedUpdate(
            set=update_set
        ).whenNotMatchedInsert(
            values=insert_values
        ).execute()
    else:
        # Target table does not exist yet — create it with initial load
        df.write.mode("overwrite").saveAsTable(target_table)

else:
    raise ValueError(f"Invalid load_type: '{load_type}'. Expected 'FULL', 'APPEND', or 'MERGE'.")

In [0]:
row_count = df.count()
dbutils.notebook.exit(str(row_count))